# ISIC-24 Modality Attendance — Post-hoc XAI & Metrics



In [ ]:
import os
import tensorflow as tf
import kagglesdk.kaggle_env as kaggle_env
import keras
import keras_cv

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

if not hasattr(kaggle_env, "get_web_endpoint"):
    kaggle_env.get_web_endpoint = kaggle_env.get_endpoint

print("GPUs:", gpus)

In [ ]:
class CFG:
    verbose = 2  # Verbosity
    seed = 49  # Random seed
    neg_sample = 0.01 # Downsample negative class
    pos_sample = 5.0  # Upsample positive class
    preset = "efficientnetv2_b2_imagenet"  # Name of pretrained classifier
    image_size = [256, 256]  # Input image size
    epochs = 20 # Training epochs
    batch_size = 64  # Batch size
    lr_mode = "cos" # LR scheduler mode from one of "cos", "step", "exp"
    class_names = ['target']
    num_classes = 1

keras.utils.set_random_seed(CFG.seed)

In [ ]:
BASE_PATH = "/mnt/disk1/home/cgiovannoni/isic24-dum"
DATA_PATH = f"{BASE_PATH}/data/isic24"
PROC_PATH = f"{BASE_PATH}/models/enhanced/enhanced/preprocessed_dataset"
ART_PATH = f"{BASE_PATH}/artifacts/enhanced"
FEATURE_SPACE_PATH = f"{ART_PATH}/feature_space"
RUNS_PATH = f"{ART_PATH}/runs"
SUMMARIES_PATH = f"{ART_PATH}/summaries"

In [ ]:
# Make the local package + the guard module importable (works with or without `pip install -e .`).
import sys, os
for _p in (os.path.join(BASE_PATH, "src"),
           os.path.join(BASE_PATH, "notebooks", "modality_attendance")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Shared modality-attendance custom layers. Importing registers the keras serializables,
# so keras.models.load_model deserializes every strategy checkpoint with NO custom_objects.
from isic24.mod_attendance import (
    ConstantScale, ScalarMultiply, LearnableAlpha, FusionWithGamma, GateEntropyLoss,
)
# FeatureSpace anti-drift guard (post-hoc correctness).
from feature_space_guard import assert_feature_space_canonical, assert_fs_matches_model


In [ ]:
# Common libraries used across the post-hoc pipeline.
import glob, gc, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

## ▶ PARAMETERS

In [ ]:
# Canonical study (timestamp dir) per strategy family. Point these elsewhere to analyse
# a different run; keep only the families you care about.
CANONICAL = {
    "vanilla":             os.path.join(RUNS_PATH, "vanilla", "20260604-210304"),
    "fixed_scaling":       os.path.join(RUNS_PATH, "fixed_scaling", "20260622-112038"),
    "learnable_scaling":   os.path.join(RUNS_PATH, "learnable_scaling", "20260619-184157"),
    "loss_regularization": os.path.join(RUNS_PATH, "loss_regularization", "20260610-095420"),
    "controlled_fusion":   os.path.join(RUNS_PATH, "controlled_fusion", "20260616-102253"),
    "adaptive_gating":     os.path.join(RUNS_PATH, "adaptive_gating", "20260629-185216"),
}
cf_study_dir = CANONICAL["controlled_fusion"]   # gamma sweep writes here
ag_study_dir = CANONICAL["adaptive_gating"]     # gate diagnostics write here

# Faithfulness budgets — TWO distinct knobs:
#   FAITH_BUDGETS = q-points the model is RE-SCORED at (the expensive/GPU knob). Keep 0.20
#     as an exact grid point so the low-budget cutoff is exact.
FAITH_BUDGETS = (0.05, 0.10, 0.20, 0.40, 0.60)
#   Q_MAX_REPORT = integration cutoff(s) defining the regime (low vs full). FREE to change:
#     re-integrates the SAVED curves, no model calls, no GPU.
Q_MAX_REPORT = [0.20, 0.60]

# --- evaluation set for the post-hoc metrics (identical for EVERY model) ---
#   "canonical" = the fold-0 set the runs were evaluated on (reproduces their numbers).
#   "custom"    = folds in EVAL_FOLDS, applied equally to all models (metric-stability test).
#   NB folds 1-4 are TRAINING data for every run, so a custom set mixes IN-SAMPLE data:
#      valid as a metric-stability stress test, NOT as held-out evaluation.
#      On a custom set only IMAGE faithfulness + the on-the-fly diagnostics run; SHAP-based
#      tabular/joint faithfulness is disabled (its saved arrays are keyed to fold-0 ordering).
EVAL_SET   = "canonical"     # "canonical" | "custom"
EVAL_FOLDS = [0, 1]          # used only when EVAL_SET == "custom"
# Faithfulness sample cap. Gate diag / gamma sweep / calibration iterate the WHOLE eval set;
# faithfulness uses only the first FAITH_N_SAMPLES rows, so raise this to enlarge IT.
FAITH_N_SAMPLES = 128

# Which runs the faithfulness loop processes (see the faithfulness cell for how they are used).
FAITH_ONLY_IMPL = None    # e.g. ["vanilla", "adaptive_gating"]; None = all discovered
FAITH_MAX_RUNS  = None    # e.g. 5 for a smoke test
FAITH_OVERWRITE = False   # False = reuse inline-computed faithfulness.json when present
FAITH_DRY_RUN   = False   # True = list what would run, load no models

# Small CSV summaries (no models loaded here).
cf_combined = pd.read_csv(os.path.join(cf_study_dir, "combined_summary.csv"))
ag_combined = pd.read_csv(os.path.join(ag_study_dir, "combined_summary.csv"))
print("params set | cf_combined", cf_combined.shape, "| ag_combined", ag_combined.shape)


## Setup — data, FeatureSpace, XAI cores

In [ ]:
import pandas as pd

# Train + Valid
df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_train_metadata_clean.parquet').ffill()

# Testing
testing_df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_test_metadata_clean.parquet').ffill()

In [ ]:
print("Class Distribution Before Sampling (%):")
print(df.target.value_counts(normalize=True)*100)

# Sampling
majority_df = df.query("target==0").sample(frac=CFG.neg_sample, random_state=CFG.seed)
minority_df = df.query("target==1").sample(frac=CFG.pos_sample, replace=True, random_state=CFG.seed)
df = pd.concat([majority_df, minority_df], axis=0).sample(frac=1.0)

print("\nClass Distribution After Sampling (%):")
print(df.target.value_counts(normalize=True)*100)

In [ ]:
import h5py

training_validation_hdf5 = h5py.File(f"{DATA_PATH}/train-image.hdf5", 'r')
testing_hdf5 = h5py.File(f"{DATA_PATH}/test-image.hdf5", 'r')

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

df = df.reset_index(drop=True) # ensure continuous index
df["fold"] = -1
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
for i, (training_idx, validation_idx) in enumerate(sgkf.split(df, y=df.target, groups=df.patient_id)):
    df.loc[validation_idx, "fold"] = int(i)

# Use first fold for training and validation
training_df = df.query("fold!=0")
validation_df = df.query("fold==0")
print(f"# Num Train: {len(training_df)} | Num Valid: {len(validation_df)}")

In [ ]:
# Categorical features which will be one hot encoded
CATEGORICAL_COLUMNS = ["sex", "anatom_site_general", "tbp_tile_type",
                       "tbp_lv_location", "attribution"]

NEW_CATEGORICAL_COLUMNS = ["combined_anatomical_site"]

# Numerical features which will be normalized
NUMERIC_COLUMNS = ["age_approx", "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
           "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio", "tbp_lv_color_std_mean",
           "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM"]

NEW_NUMERIC_COLUMNS = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
         "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency", "color_consistency",

        "size_age_interaction", "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation", "overall_color_difference",
        "symmetry_perimeter_interaction", "comprehensive_lesion_index"]

# Tabular feature columns
FEAT_COLS = (CATEGORICAL_COLUMNS + NEW_CATEGORICAL_COLUMNS) + (NUMERIC_COLUMNS + NEW_NUMERIC_COLUMNS)

In [ ]:
def build_augmenter():
    aug_layers = [
        keras_cv.layers.RandomCutout(height_factor=(0.02, 0.06), width_factor=(0.02, 0.06)),
        keras_cv.layers.RandomFlip(mode="horizontal"),
    ]
    aug_layers = [keras_cv.layers.RandomApply(x, rate=0.5) for x in aug_layers]
    augmenter = keras_cv.layers.Augmenter(aug_layers)

    def augment(inp, label):
        images = inp["images"]
        aug_data = {"images": images}
        aug_data = augmenter(aug_data)
        inp["images"] = aug_data["images"]
        return inp, label
    return augment


def build_decoder(with_labels=True, target_size=CFG.image_size):
    def decode_image(inp):
        file_bytes = inp["images"]
        image = tf.io.decode_jpeg(file_bytes)
        image = tf.image.resize(image, size=target_size, method="area")
        image = tf.cast(image, tf.float32) / 255.0
        image = tf.reshape(image, [*target_size, 3])
        inp["images"] = image
        return inp

    def decode_label(label, num_classes):
        label = tf.cast(label, tf.float32)
        label = tf.reshape(label, [num_classes])
        return label

    def decode_with_labels(inp, label=None):
        inp = decode_image(inp)
        label = decode_label(label, CFG.num_classes)
        return (inp, label)

    return decode_with_labels if with_labels else decode_image


def build_dataset(
    isic_ids,
    hdf5,
    features,
    labels=None,
    batch_size=32,
    decode_fn=None,
    augment_fn=None,
    augment=False,
    shuffle=1024,
    cache=True,
    drop_remainder=False,
):
    if decode_fn is None:
        decode_fn = build_decoder(labels is not None)
    if augment_fn is None:
        augment_fn = build_augmenter()

    AUTO = tf.data.AUTOTUNE

    images = [None]*len(isic_ids)
    for i, isic_id in enumerate(tqdm(isic_ids, desc="Loading Images ")):
        images[i] = hdf5[isic_id][()]

    inp = {"images": images, "features": features}
    slices = (inp, labels) if labels is not None else inp

    ds = tf.data.Dataset.from_tensor_slices(slices)

    # Decode before cache (so we don't decode every epoch)
    ds = ds.map(decode_fn, num_parallel_calls=AUTO)

    if cache:
        ds = ds.cache()

    if shuffle:
        ds = ds.shuffle(shuffle, seed=CFG.seed, reshuffle_each_iteration=True)
        opt = tf.data.Options()
        opt.experimental_deterministic = False
        opt.experimental_slack = True
        ds = ds.with_options(opt)

    ds = ds.batch(batch_size, drop_remainder=drop_remainder)

    # Keep augmentation AFTER cache so it's still random per epoch
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=AUTO)

    ds = ds.prefetch(AUTO)
    return ds

def _to_feature_dict(df):
    feat_dict = {}
    for c in df.columns:
        s = df[c]
        if s.dtype == object or str(s.dtype).startswith('string'):
            feat_dict[c] = s.astype(str).to_numpy()
        else:
            feat_dict[c] = s.to_numpy()
    return feat_dict

In [ ]:
from tqdm import tqdm

# Build datasets
print("# Training:")
training_features = _to_feature_dict(training_df[FEAT_COLS])
training_ids = training_df.isic_id.values
training_labels = training_df.target.values
training_ds = build_dataset(
    training_ids, training_validation_hdf5, training_features, training_labels,
    batch_size=CFG.batch_size, shuffle=True, augment=True
)

print("# Validation:")
validation_features = _to_feature_dict(validation_df[FEAT_COLS])
validation_ids = validation_df.isic_id.values
validation_labels = validation_df.target.values
validation_ds = build_dataset(
    validation_ids, training_validation_hdf5, validation_features, validation_labels,
    batch_size=CFG.batch_size, shuffle=False, augment=False
)

print("# Testing:")
testing_features = _to_feature_dict(testing_df[FEAT_COLS])
testing_ids = testing_df.isic_id.values
testing_ds = build_dataset(
    testing_ids, testing_hdf5, testing_features,
    batch_size=CFG.batch_size, shuffle=False, augment=False, cache=False
)

In [ ]:
# Evaluation set for the post-hoc metrics + validation-set identity guard.
EVAL_IS_CANONICAL = (EVAL_SET == "canonical")
if EVAL_IS_CANONICAL:
    _val_snap = os.path.join(BASE_PATH, "notebooks", "modality_attendance",
                             "validation_ids.SEED49.json")
    if os.path.exists(_val_snap):
        _snap = json.load(open(_val_snap))
        _match = (sorted(validation_df.isic_id.tolist()) == _snap["isic_ids_sorted"])
        # WARN (not raise): the shipped snapshot is a post-hoc regen, deterministic but NOT
        # verified against the runs' training split. Flip to `raise` once a real training run
        # has (re)written it (source='training_run').
        print(f"[EVAL GUARD] validation set vs snapshot (source={_snap.get('source','?')}): "
              f"{'MATCH' if _match else 'MISMATCH'} "
              f"(have {len(validation_df)}, snapshot {_snap['n']}).")
        if not _match:
            print("  \u26a0\ufe0f  reconstructed fold-0 set differs from the snapshot. If the snapshot came "
                  "from the real training run, post-hoc numbers will NOT correspond to the checkpoints.")
    else:
        print(f"[EVAL GUARD] no snapshot at {_val_snap}; run the training notebook once to write it.")
else:
    validation_df = df.query("fold in @EVAL_FOLDS").reset_index(drop=True)
    validation_ds = build_dataset(
        validation_df.isic_id.values, training_validation_hdf5,
        _to_feature_dict(validation_df[FEAT_COLS]), validation_df.target.values,
        batch_size=CFG.batch_size, shuffle=False, augment=False)
    print(f"[EVAL] custom set folds={EVAL_FOLDS}: {len(validation_df)} samples. "
          "SHAP tabular/joint faithfulness disabled; image faithfulness + on-the-fly diagnostics only.")


In [ ]:
feature_space = keras.utils.FeatureSpace(
    features={
        # Categorical features encoded as integers
        "sex": "string_categorical",
        "anatom_site_general": "string_categorical",
        "tbp_tile_type": "string_categorical",
        "tbp_lv_location": "string_categorical",
        "combined_anatomical_site": "string_categorical",
        "attribution": "string_categorical",
        # Numerical features to discretize
        "age_approx": "float_discretized",
        # Numerical features to normalize
        "tbp_lv_nevi_confidence": "float_normalized",
        "clin_size_long_diam_mm": "float_normalized",
        "tbp_lv_areaMM2": "float_normalized",
        "tbp_lv_area_perim_ratio": "float_normalized",
        "tbp_lv_color_std_mean": "float_normalized",
        "tbp_lv_deltaLBnorm": "float_normalized",
        "tbp_lv_minorAxisMM": "float_normalized",
        "lesion_size_ratio" : "float_normalized",
        "lesion_shape_index" : "float_normalized",
        "hue_contrast" : "float_normalized",
        "luminance_contrast": "float_normalized",
        "lesion_color_difference": "float_normalized",
        "border_complexity": "float_normalized",
        "3d_position_distance": "float_normalized",
        "perimeter_to_area_ratio": "float_normalized",
        "lesion_visibility_score": "float_normalized",
        "symmetry_border_consistency": "float_normalized",
        "color_consistency": "float_normalized",
        "size_age_interaction": "float_normalized",
        "hue_color_std_interaction": "float_normalized",
        "lesion_severity_index": "float_normalized",
        "shape_complexity_index": "float_normalized",
        "color_contrast_index": "float_normalized",
        "log_lesion_area": "float_normalized",
        "mean_hue_difference": "float_normalized",
        "std_dev_contrast": "float_normalized",
        "color_shape_composite_index": "float_normalized",
        "3d_lesion_orientation": "float_normalized",
        "overall_color_difference": "float_normalized",
        "symmetry_perimeter_interaction": "float_normalized",
        "comprehensive_lesion_index": "float_normalized",
        # numerical features already normalized
        "normalized_lesion_size": "float",
        },
    output_mode="concat",
)

# ---------------------------------------------------------------------------
# REPRODUCIBILITY FIX (2026-07-14).
# `training_ds` is built with shuffle=True, reshuffle_each_iteration=True and
# experimental_deterministic=False. Discretization (streaming quantile sketch) and
# StringLookup (vocab ordered by frequency) are ORDER-DEPENDENT, so adapt() produced
# slightly different statistics on every notebook session -- which is why reloading a
# checkpoint post-hoc does not reproduce the cached predictions (the error grows with
# alpha_tab: exact at alpha=0, max |dp| = 0.24 at alpha=1).
#
# Load the saved FeatureSpace when it exists; only adapt (and save) when it does not.
# Set FORCE_READAPT_FEATURE_SPACE = True to deliberately re-adapt.
# ---------------------------------------------------------------------------
FORCE_READAPT_FEATURE_SPACE = False

_fs_file = os.path.join(FEATURE_SPACE_PATH, "feature_space.keras")

if os.path.exists(_fs_file) and not FORCE_READAPT_FEATURE_SPACE:
    feature_space = keras.saving.load_model(_fs_file)
    print(f"FeatureSpace LOADED from {_fs_file} (deterministic; not re-adapted)")
else:
    training_ds_with_no_labels = training_ds.map(lambda x, _: x["features"])
    feature_space.adapt(training_ds_with_no_labels)
    os.makedirs(FEATURE_SPACE_PATH, exist_ok=True)
    feature_space.save(_fs_file)
    print(f"FeatureSpace ADAPTED and saved to {_fs_file}")

print(feature_space.get_encoded_features().shape)

In [ ]:
# FeatureSpace anti-drift guard: fail LOUD if the live FeatureSpace is not the canonical one
# the checkpoints were trained with (dim 127 / recorded fingerprint).
assert_feature_space_canonical(feature_space)


In [ ]:
# Apply Feature Processing
augment_fn = build_augmenter()

auto = tf.data.AUTOTUNE

training_ds = training_ds.map(
    lambda x, y: ({'images': x['images'], 'features': feature_space(x['features'])}, y),
    num_parallel_calls=auto,
)
training_ds = training_ds.map(augment_fn, num_parallel_calls=auto)
training_ds = training_ds.prefetch(auto)

validation_ds = validation_ds.map(
    lambda x, y: ({'images': x['images'], 'features': feature_space(x['features'])}, y),
    num_parallel_calls=auto,
)
validation_ds = validation_ds.prefetch(auto)

testing_ds = testing_ds.map(
    lambda x: {'images': x['images'], 'features': feature_space(x['features'])},
    num_parallel_calls=auto,
)
testing_ds = testing_ds.prefetch(auto)

In [ ]:
# optimizer, loss & metric
adam = keras.optimizers.Adam(learning_rate=1e-4)
auc = keras.metrics.AUC(name='auc')
bce = keras.losses.BinaryCrossentropy(label_smoothing=0.02)

In [ ]:
_ = feature_space.get_encoded_features()

# Get the feature order as used by FeatureSpace
try:
    feature_order = list(feature_space.features.keys())
except AttributeError:
    # Fallback: you know the order from how you created the FeatureSpace
    feature_order = [
        "sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location",
        "combined_anatomical_site", "attribution",
        "age_approx",
        "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
        "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio",
        "tbp_lv_color_std_mean", "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM",
        "lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
        "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency",
        "color_consistency", "size_age_interaction",
        "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation",
        "overall_color_difference", "symmetry_perimeter_interaction",
        "comprehensive_lesion_index",
        "normalized_lesion_size",
    ]

In [ ]:
from keras.layers import StringLookup, Discretization, Normalization

encoded_feature_names = []
feature_dim_sizes = {}

for name in feature_order:
    preproc = feature_space.preprocessors[name]

    # 1) Categorical features: StringLookup → one-hot of size vocabulary_size
    if isinstance(preproc, StringLookup):
        vocab = preproc.get_vocabulary()
        # each token gets its own encoded dim
        for token in vocab:
            encoded_feature_names.append(f"{name}__{token}")
        d = len(vocab)

    # 2) Discretized float (age_approx): Discretization → one-hot over bins
    elif isinstance(preproc, Discretization):
        boundaries = preproc.get_config().get("bin_boundaries", [])
        n_bins = len(boundaries) + 1
        for b in range(n_bins):
            encoded_feature_names.append(f"{name}_bin_{b}")
        d = n_bins

    # 3) Normalized numeric or raw float: 1 dim
    elif isinstance(preproc, Normalization):
        encoded_feature_names.append(name)
        d = 1

    else:
        # "float" (already normalized) or any other scalar
        encoded_feature_names.append(name)
        d = 1

    feature_dim_sizes[name] = d

# sanity check
print("per-feature dim sizes:", feature_dim_sizes)
print("len(encoded_feature_names):", len(encoded_feature_names))
print("feature_space encoded dim:", feature_space.get_encoded_features().shape[-1])

len(encoded_feature_names) == feature_space.get_encoded_features().shape[-1] == 128

In [ ]:
# ================================================================================
# XAI engines + faithfulness — imported, not defined here.
# --------------------------------------------------------------------------------
# Grad-CAM, the faithfulness machinery and the SHAP/LORE engines used to be ~1400 lines
# inline here AND byte-duplicated in the training notebook. They now live in
# `src/isic24/`, so both notebooks drive the SAME code.
#
# This also fixes a latent ordering bug: the faithfulness cell called
# `encoded_indices_for_raw` and `_finalize_plot`, which the notebook only defined further
# DOWN, in the archived appendix — running this notebook linearly raised NameError on the
# tabular method.
# ================================================================================
from isic24 import xai_faithfulness as xf
from isic24.xai_faithfulness import (
    explanation_faithfulness, faith_subsets, build_strata, persample_gains,
    stratified_table, pool_by_family, discover_curve_files, verify_reintegration,
    _bootstrap_ci, _build_faith_subset, _load_shap_values, _encode_raw, _score_logit,
)
from isic24.xai_driver import run_posthoc_xai, run_posthoc_xai_batch, make_xai_callbacks
from isic24.xai_engines import (
    MultiModalGradCAMEngine, MultiModalGradCAMCallback,
    encoded_indices_for_raw, aggregate_raw_importance, _finalize_plot, PLOT_SHOW,
)

# Hand the notebook's data objects to the faithfulness code once, plus the knobs from the
# PARAMETERS cell (they used to be read out of the caller's globals()).
xf.configure(
    xf.XaiContext(
        feat_cols=FEAT_COLS,
        training_df=training_df,
        validation_df=validation_df,
        validation_ds=validation_ds,
        feature_space=feature_space,
        to_feature_dict=_to_feature_dict,
        encoded_feature_names=encoded_feature_names,
    ),
    FAITH_N_SAMPLES=FAITH_N_SAMPLES,
    FAITH_BUDGETS=tuple(FAITH_BUDGETS),
    EVAL_IS_CANONICAL=EVAL_IS_CANONICAL,
)
print("[XAI] engines imported | eval set:", EVAL_SET, "| n_faith:", FAITH_N_SAMPLES)


In [ ]:
# Post-hoc plotting + stale-path helpers.

def relmap(rd):
    # combined_summary.csv keeps the training machine's absolute paths; remap the '/runs/...'
    # suffix onto this machine's RUNS_PATH so checkpoints/curves resolve.
    if os.path.isdir(rd):
        return rd
    s = str(rd).replace("\\", "/"); i = s.find("/runs/")
    return os.path.join(os.path.dirname(RUNS_PATH), s[i + 1:]) if i >= 0 else rd


## Post-hoc diagnostics — gate faithfulness, calibration, gamma sweep

In [ ]:
# Generic AUC evaluator on a dataset
def eval_auc(model, ds):
    m = keras.metrics.AUC()
    for x, y in ds:
        p = model(x, training=False)
        m.update_state(y, p)
    return float(m.result().numpy())

# Build ablated inputs to remove/perturb one modality
def ablate_batch(x, mode, strength):
    x = dict(x)
    if mode in ("img", "both"):
        img = tf.cast(x["images"], tf.float32)
        x["images"] = (1.0 - strength) * img + strength * (tf.ones_like(img) * 0.5)  # gray image
    if mode in ("tab", "both"):
        feat = tf.cast(x["features"], tf.float32)
        mask = tf.random.uniform(tf.shape(feat)) >= strength
        x["features"] = tf.where(mask, feat, tf.zeros_like(feat))
    return x

# Evaluate AUC after modality ablation
def eval_auc_with_ablation(model, ds, mode, strength):
    m = keras.metrics.AUC()
    for x, y in ds:
        p = model(ablate_batch(x, mode, strength), training=False)
        m.update_state(y, p)
    return float(m.result().numpy())

# Simple Spearman (rank correlation) implementation
def spearman(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    ra = a.argsort().argsort().astype(np.float32)
    rb = b.argsort().argsort().astype(np.float32)
    ra -= ra.mean()
    rb -= rb.mean()
    return float((ra @ rb) / (np.linalg.norm(ra) * np.linalg.norm(rb) + 1e-12))

# Convert probabilities to logits for impact comparison
def logit(p, eps=1e-6):
    p = tf.clip_by_value(tf.cast(p, tf.float32), eps, 1 - eps)
    return tf.math.log(p / (1 - p))

# Gate-faithfulness: do gate weights correlate with true modality impact?
def gate_faithfulness(model, ds, strength=1.0):
    gate_model = keras.Model(model.inputs, model.get_layer("gate_weights").output)

    wimg, wtab, dimg, dtab = [], [], [], []
    for x, _ in ds:
        p0 = tf.squeeze(model(x, training=False), -1)
        l0 = logit(p0)

        # Impact after image ablation
        pi = tf.squeeze(model(ablate_batch(x, "img", strength), training=False), -1)
        di = tf.abs(l0 - logit(pi))

        # Impact after tabular ablation
        pt = tf.squeeze(model(ablate_batch(x, "tab", strength), training=False), -1)
        dt = tf.abs(l0 - logit(pt))

        # Predicted gate weights
        w = gate_model(x, training=False)
        wimg.append(w[:, 0].numpy())
        wtab.append(w[:, 1].numpy())
        dimg.append(di.numpy())
        dtab.append(dt.numpy())

    wimg = np.concatenate(wimg)
    wtab = np.concatenate(wtab)
    dimg = np.concatenate(dimg)
    dtab = np.concatenate(dtab)

    return {
        "spearman_w_img_delta_img": spearman(wimg, dimg),
        "spearman_w_tab_delta_tab": spearman(wtab, dtab),
    }

# Calibration metrics: Brier score + ECE
def brier_ece(model, ds, n_bins=15):
    y_all, p_all = [], []
    for x, y in ds:
        p = tf.squeeze(model(x, training=False), -1)
        y_all.append(tf.cast(tf.squeeze(y), tf.float32).numpy())
        p_all.append(tf.cast(p, tf.float32).numpy())

    y = np.concatenate(y_all)
    p = np.concatenate(p_all)
    brier = float(np.mean((p - y) ** 2))

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    N = len(p)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if np.any(m):
            conf = float(np.mean(p[m]))
            acc = float(np.mean(y[m]))
            ece += (np.sum(m) / N) * abs(acc - conf)

    return {"brier": brier, "ece": float(ece)}

In [ ]:
# post-training diagnostics

all_diag = []
for _, row in ag_combined.iterrows():
    ag_run_dir = row["run_dir"]
    # combined_summary.csv stores June paths (/home/claudio/...); remap the
    # "/runs/..." suffix onto this machine's RUNS_PATH so checkpoints load.
    if not os.path.isdir(ag_run_dir):
        _s = str(ag_run_dir).replace("\\", "/")
        _i = _s.find("/runs/")
        if _i >= 0:
            ag_run_dir = os.path.join(os.path.dirname(RUNS_PATH), _s[_i + 1:])
    best_ckpt = os.path.join(ag_run_dir, "checkpoints", f"best_{row['model_name']}.keras")
    ag_model = keras.models.load_model(best_ckpt, compile=False, safe_mode=False)
    assert_fs_matches_model(ag_model, feature_space, row.get("run_id"))

    gate_model = keras.Model(ag_model.inputs, ag_model.get_layer("gate_weights").output)
    gx, _ = next(iter(validation_ds))
    gw = gate_model(gx, training=False).numpy()

    base_auc = eval_auc(ag_model, validation_ds)
    img_auc = eval_auc_with_ablation(ag_model, validation_ds, "img", 1.0)
    tab_auc = eval_auc_with_ablation(ag_model, validation_ds, "tab", 1.0)

    diag = {
        "run_id": row["run_id"],
        "setting_tag": row["setting_tag"],
        "seed": int(row["seed"]),
        "entropy_weight": float(row["entropy_weight"]),
        "run_dir": ag_run_dir,
        "gate_weight_mean_img": float(np.mean(gw[:, 0])),
        "gate_weight_mean_tab": float(np.mean(gw[:, 1])),
        "base_auc": base_auc,
        "img_drop_auc": base_auc - img_auc,
        "tab_drop_auc": base_auc - tab_auc,
        **gate_faithfulness(ag_model, validation_ds, strength=1.0),
        **brier_ece(ag_model, validation_ds),
    }

    os.makedirs(os.path.join(ag_run_dir, "diagnostics"), exist_ok=True)
    with open(os.path.join(ag_run_dir, "diagnostics", "adaptive_gating_diagnostics.json"), "w") as f:
        json.dump(diag, f, indent=2)
    all_diag.append(diag)

    # ---- diagnostic plots (per run), saved to the run's plots/ folder ----
    plots_dir = os.path.join(ag_run_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    # gather full-validation gate weights, labels and predictions in one pass
    wimg_all, wtab_all, y_true_all, p_pred_all = [], [], [], []
    for xb, yb in validation_ds:
        wb = gate_model(xb, training=False).numpy()
        pb = tf.squeeze(ag_model(xb, training=False), -1).numpy()
        wimg_all.append(wb[:, 0]); wtab_all.append(wb[:, 1])
        y_true_all.append(np.asarray(tf.squeeze(yb)).reshape(-1))
        p_pred_all.append(np.asarray(pb).reshape(-1))
    wimg_all = np.concatenate(wimg_all); wtab_all = np.concatenate(wtab_all)
    y_true_all = np.concatenate(y_true_all); p_pred_all = np.concatenate(p_pred_all)

    # 1) gate-weight distribution
    plt.figure(figsize=(7, 4))
    plt.hist(wimg_all, bins=30, alpha=0.6, label="w_img")
    plt.hist(wtab_all, bins=30, alpha=0.6, label="w_tab")
    plt.xlabel("gate weight"); plt.ylabel("count")
    plt.title(f"Adaptive gating: gate-weight distribution (seed {int(row['seed'])})")
    plt.legend(); plt.grid(alpha=0.3)
    _finalize_plot(plots_dir, "gate_weight_distribution.png")

    # 2) modality ablation: base AUC and AUC drop after ablating each modality
    plt.figure(figsize=(7, 4))
    labels = ["base_auc", "img_drop_auc", "tab_drop_auc"]
    vals = [diag["base_auc"], diag["img_drop_auc"], diag["tab_drop_auc"]]
    plt.bar(labels, vals, color=["#4c72b0", "#dd8452", "#55a868"])
    for j, v in enumerate(vals):
        plt.text(j, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    plt.ylabel("AUC"); plt.grid(alpha=0.3, axis="y")
    plt.title(f"Adaptive gating: modality ablation (seed {int(row['seed'])})")
    _finalize_plot(plots_dir, "modality_ablation_auc_drop.png")

    # 3) calibration reliability diagram
    n_bins = 15
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_conf, bin_obs = [], []
    for j in range(n_bins):
        lo, hi = edges[j], edges[j + 1]
        m = (p_pred_all >= lo) & (p_pred_all < hi) if j < n_bins - 1 else (p_pred_all >= lo) & (p_pred_all <= hi)
        if np.any(m):
            bin_conf.append(float(np.mean(p_pred_all[m])))
            bin_obs.append(float(np.mean(y_true_all[m])))
    plt.figure(figsize=(5, 5))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="perfect")
    plt.plot(bin_conf, bin_obs, marker="o", label="model")
    plt.xlabel("mean predicted probability"); plt.ylabel("observed frequency")
    plt.title(f"Calibration (Brier={diag['brier']:.3f}, ECE={diag['ece']:.3f})")
    plt.legend(); plt.grid(alpha=0.3)
    _finalize_plot(plots_dir, "calibration_reliability.png")

    keras.backend.clear_session()
    gc.collect()

ag_diag_df = pd.DataFrame(all_diag)
ag_diag_df.to_csv(os.path.join(ag_study_dir, "adaptive_gating_diagnostics.csv"), index=False)
display(ag_diag_df)

In [ ]:
# Post-hoc gamma sweep over every controlled-fusion run
# gamma_tab > 1 enhances tabular / penalizes image (gamma_img < 1)
# gamma_tab < 1 penalizes tabular / enhances image (gamma_img > 1)
DELTAS = np.round(np.arange(-0.5, 0.501, 0.125), 3)
POSTHOC_SETTINGS = []
for d in DELTAS:
    POSTHOC_SETTINGS.append({
        "tag": f"d{d:+.3f}".replace(".", "p").replace("+", "plus").replace("-", "minus"),
        "delta": float(d),
        "gamma_img": float(1.0 - d),
        "gamma_tab": float(1.0 + d),
        "mode": "tab_enhance_img_penalize" if d > 0 else ("tab_penalize_img_enhance" if d < 0 else "neutral"),
    })

posthoc_rows = []
for _, row in cf_combined.iterrows():
    cf_run_dir = row["run_dir"]
    ckpt_path = os.path.join(cf_run_dir, "checkpoints", f"best_{row['model_name']}.keras")
    best_model = keras.models.load_model(ckpt_path, compile=False, safe_mode=False)
    assert_fs_matches_model(best_model, feature_space, row.get("run_id"))
    fusion_layer = best_model.get_layer("fusion_with_gamma")

    run_rows = []
    for s in POSTHOC_SETTINGS:
        fusion_layer.gamma_img.assign(s["gamma_img"])
        fusion_layer.gamma_tab.assign(s["gamma_tab"])
        auc_val = eval_auc(best_model, validation_ds)

        rec = {
            "run_id": row["run_id"],
            "setting_tag": row["setting_tag"],
            "seed": int(row["seed"]),
            "run_dir": cf_run_dir,
            "posthoc_id": s["tag"],
            "delta": s["delta"],
            "gamma_img": s["gamma_img"],
            "gamma_tab": s["gamma_tab"],
            "mode": s["mode"],
            "val_auc": auc_val,
            "base_model_name": row["model_name"],
        }
        run_rows.append(rec)
        posthoc_rows.append(rec)

    posthoc_df = pd.DataFrame(run_rows).sort_values("gamma_tab").reset_index(drop=True)
    diagnostics_dir = os.path.join(cf_run_dir, "diagnostics")
    os.makedirs(diagnostics_dir, exist_ok=True)
    posthoc_df.to_csv(os.path.join(diagnostics_dir, "posthoc_gamma_summary.csv"), index=False)

    plots_dir = os.path.join(cf_run_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    plt.figure(figsize=(7, 4))
    plt.plot(posthoc_df["gamma_tab"], posthoc_df["val_auc"], marker="o")
    plt.axvline(1.0, linestyle="--", linewidth=1)
    plt.xlabel("gamma_tab")
    plt.ylabel("val_auc")
    plt.title(f"Controlled fusion post-hoc: val_auc vs gamma_tab (seed {int(row['seed'])})")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "posthoc_val_auc_vs_gamma_tab.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(7, 4))
    plt.scatter(posthoc_df["gamma_img"], posthoc_df["gamma_tab"], c=posthoc_df["val_auc"])
    for _, r in posthoc_df.iterrows():
        plt.text(r["gamma_img"], r["gamma_tab"], f"{r['val_auc']:.3f}", fontsize=8)
    plt.xlabel("gamma_img")
    plt.ylabel("gamma_tab")
    plt.title(f"Controlled fusion post-hoc sweep (seed {int(row['seed'])})")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "posthoc_gamma_sweep_scatter.png"), dpi=300)
    plt.close()

    keras.backend.clear_session()
    gc.collect()

posthoc_all_df = pd.DataFrame(posthoc_rows).sort_values(["seed", "gamma_tab"]).reset_index(drop=True)
posthoc_all_df.to_csv(os.path.join(cf_study_dir, "controlled_fusion_posthoc_gamma_summary.csv"), index=False)

posthoc_agg = (
    posthoc_all_df.groupby(["delta", "gamma_img", "gamma_tab"], dropna=False)["val_auc"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
posthoc_agg.to_csv(os.path.join(cf_study_dir, "controlled_fusion_posthoc_gamma_aggregated.csv"), index=False)
display(posthoc_all_df.head(20))
display(posthoc_agg)

## Explanation Faithfulness (Deletion / Insertion AUC)

`faithfulness_gain = deletion_auc_explanation - deletion_auc_random` (piu alto = meglio), calcolata per `tabular` (SHAP), `image` (Grad-CAM) e `joint`. Riusa gli array SHAP gia salvati (nessun re-fit del GradientExplainer). Le tre celle seguenti definiscono il core, lo eseguono post-hoc su tutti i run gia allenati e aggregano i risultati.

In [ ]:
# ---------------------------------------------------------------------
# Post-hoc faithfulness over ALL trained runs (no retraining).
# Discovers run dirs from single_run_summary.csv, loads each best_*.keras
# checkpoint, computes deletion/insertion faithfulness and saves per-run
# artifacts + a combined CSV.
# ---------------------------------------------------------------------
FAITH_RUN_GLOB = os.path.join(RUNS_PATH, "**", "single_run_summary.csv")  # any depth: 3-level single + 4-level multi-seed
FAITH_OUT_DIR = os.path.join(RUNS_PATH, "_faithfulness")
os.makedirs(FAITH_OUT_DIR, exist_ok=True)
_faith_name = "faithfulness" if globals().get("EVAL_IS_CANONICAL", True) else f"faithfulness_{EVAL_SET}"
_faith_methods = None if globals().get("EVAL_IS_CANONICAL", True) else ("image",)

# FAITH_ONLY_IMPL / FAITH_OVERWRITE / FAITH_MAX_RUNS / FAITH_DRY_RUN come from the PARAMETERS cell.

_SETTING_PARAM_COLS = ("alpha", "init_alpha", "lambda_reg",
                       "gamma_img", "gamma_tab", "entropy_weight")

summ_paths = sorted(glob.glob(FAITH_RUN_GLOB, recursive=True))

selected_runs = []
n_reuse = 0
n_compute = 0
for sp in summ_paths:
    try:
        row = pd.read_csv(sp).iloc[0].to_dict()
    except Exception as e:
        print("  skip (bad csv):", sp, e)
        continue

    impl = row.get("impl_name")
    if FAITH_ONLY_IMPL and impl not in FAITH_ONLY_IMPL:
        continue

    run_dir = row.get("run_dir")
    if not isinstance(run_dir, str) or not os.path.isdir(run_dir):
        run_dir = os.path.dirname(sp)
    model_name = row.get("model_name")
    best_epoch = int(row.get("best_epoch", -1))
    ckpt = os.path.join(run_dir, "checkpoints", f"best_{model_name}.keras")
    if not os.path.exists(ckpt) or best_epoch < 0:
        print("  skip (no ckpt / best_epoch):", run_dir)
        continue

    fjson = os.path.join(run_dir, "xai", "faithfulness", f"{_faith_name}.json")
    will_reuse = os.path.exists(fjson) and not FAITH_OVERWRITE
    n_reuse += int(will_reuse)
    n_compute += int(not will_reuse)
    selected_runs.append((sp, row, run_dir, model_name, best_epoch, ckpt, fjson))

if FAITH_MAX_RUNS is not None:
    selected_runs = selected_runs[:int(FAITH_MAX_RUNS)]
    n_reuse = sum(os.path.exists(r[6]) and not FAITH_OVERWRITE for r in selected_runs)
    n_compute = len(selected_runs) - n_reuse

print(f"[FAITH] discovered {len(summ_paths)} run summaries | selected={len(selected_runs)} "
      f"| reuse={n_reuse} | compute={n_compute}")
if FAITH_DRY_RUN:
    display(pd.DataFrame([{"impl_name": r[1].get("impl_name"),
                           "setting_tag": r[1].get("setting_tag"),
                           "seed": r[1].get("seed"),
                           "run_dir": r[2],
                           "action": "reuse" if os.path.exists(r[6]) and not FAITH_OVERWRITE else "compute"}
                          for r in selected_runs]).head(50))
    raise SystemExit("FAITH_DRY_RUN=True: no models loaded.")

faith_rows = []
for sp, row, run_dir, model_name, best_epoch, ckpt, fjson in selected_runs:
    impl = row.get("impl_name")
    if os.path.exists(fjson) and not FAITH_OVERWRITE:
        res = json.load(open(fjson))
        res = {k: v for k, v in res.items() if k not in ("curves", "q_grid")}
    else:
        t0 = time.time()
        try:
            keras.backend.clear_session()
            model = keras.models.load_model(ckpt, compile=False, safe_mode=False)
            assert_fs_matches_model(model, feature_space, run_dir)
            res = explanation_faithfulness(model, run_dir, best_epoch, save=True, verbose=False,
                                           out_name=_faith_name, methods=_faith_methods)
        except Exception as e:
            print(f"  [WARN] {impl}/{row.get('setting_tag')}: {type(e).__name__}: {e}")
            keras.backend.clear_session(); gc.collect()
            continue
        keras.backend.clear_session(); gc.collect()
        res["faith_seconds"] = round(time.time() - t0, 2)

    rec = {
        "impl_name": impl,
        "setting_tag": row.get("setting_tag"),
        "seed": row.get("seed"),
        "best_val_auc": row.get("best_val_auc"),
        "run_dir": run_dir,
        **{c: row[c] for c in _SETTING_PARAM_COLS if c in row},
        **{k: v for k, v in res.items() if not isinstance(v, (list, dict))},
    }
    faith_rows.append(rec)
    print(f"  ok {impl:>18}/{str(row.get('setting_tag')):<14} "
          f"gain={res.get('faithfulness_gain', float('nan')):+.4f} "
          f"({res.get('faith_seconds', '?')}s)")

faith_df = pd.DataFrame(faith_rows)
out_csv = os.path.join(FAITH_OUT_DIR, f"combined_{_faith_name}.csv")
# MERGE (not overwrite): keep prior rows for runs not recomputed in this pass,
# replace rows for run_dirs processed now. Safe with FAITH_ONLY_IMPL filtering.
if os.path.exists(out_csv) and len(faith_df):
    _prev = pd.read_csv(out_csv)
    _prev = _prev[~_prev["run_dir"].isin(faith_df["run_dir"])]
    faith_df = pd.concat([_prev, faith_df], ignore_index=True)
elif os.path.exists(out_csv) and not len(faith_df):
    faith_df = pd.read_csv(out_csv)
faith_df.to_csv(out_csv, index=False)
print(f"[FAITH] saved {out_csv}  shape={faith_df.shape}")
display(faith_df.head(20))


In [ ]:
# ---------------------------------------------------------------------
# Cross-strategy aggregation: mean/std over seeds, bootstrap CI,
# comparison plots. Run after the post-hoc loop above.
# ---------------------------------------------------------------------
faith_csv = os.path.join(RUNS_PATH, "_faithfulness", "combined_faithfulness.csv")
fdf = pd.read_csv(faith_csv)

# --- keep ONLY canonical runs: drop seed_49 smoke runs and the superseded
#     duplicate controlled_fusion run so nothing is double-counted here.
#     (CANONICAL is defined in the "canonical study dirs" cell above.)
_canon_ts = {impl: os.path.basename(p) for impl, p in CANONICAL.items()}
fdf["_ts"] = fdf["run_dir"].astype(str).str.extract(r"/(\d{8}-\d{6})/")[0]
fdf = fdf[fdf.apply(lambda r: _canon_ts.get(r["impl_name"]) == r["_ts"], axis=1)].copy()
fdf = fdf.drop(columns=["_ts"]).reset_index(drop=True)
print(f"[FAITH] aggregating over {len(fdf)} canonical runs: "
      f"{fdf['impl_name'].value_counts().to_dict()}")
FAITH_OUT_DIR = os.path.join(RUNS_PATH, "_faithfulness")
FAITH_SHOW_AGG_PLOTS = False  # save aggregate PNGs without flooding notebook output

gain_cols = [c for c in fdf.columns
             if c.startswith("faithfulness_") and c.endswith("_gain")]
val_cols = [c for c in (gain_cols + ["faithfulness_gain", "best_val_auc"]) if c in fdf.columns]

# ---- mean / std / n per (impl_name, setting_tag) over seeds ----
agg = (fdf.groupby(["impl_name", "setting_tag"], dropna=False)[val_cols]
          .agg(["mean", "std", "count"]))
agg.to_csv(os.path.join(FAITH_OUT_DIR, "faithfulness_summary.csv"))
print("[FAITH] per-setting summary:")
display(agg)

# ---- bootstrap CI of primary faithfulness_gain per strategy (resample runs) ----
ci_rows = []
for impl, sub in fdf.groupby("impl_name"):
    m, lo, hi = _bootstrap_ci(sub["faithfulness_gain"].values)
    ci_rows.append({"impl_name": impl, "mean_gain": m, "ci_lo": lo, "ci_hi": hi, "n_runs": len(sub)})
ci_df = pd.DataFrame(ci_rows).sort_values("mean_gain", ascending=False).reset_index(drop=True)
ci_df.to_csv(os.path.join(FAITH_OUT_DIR, "faithfulness_gain_ci.csv"), index=False)
print("[FAITH] strategy ranking (primary deletion gain, bootstrap 95% CI):")
display(ci_df)

# ---- bar chart: faithfulness_gain per strategy with CI ----
plt.figure(figsize=(9, 5))
x = np.arange(len(ci_df))
yerr = [ci_df["mean_gain"] - ci_df["ci_lo"], ci_df["ci_hi"] - ci_df["mean_gain"]]
plt.bar(x, ci_df["mean_gain"], yerr=yerr, capsize=4, color="#4c72b0", edgecolor="black")
plt.axhline(0.0, color="gray", lw=1)
plt.xticks(x, ci_df["impl_name"], rotation=30, ha="right")
plt.ylabel("faithfulness_gain (primary deletion)")
plt.title("Explanation faithfulness gain per strategy (bootstrap 95% CI)")
_finalize_plot(FAITH_OUT_DIR, "faithfulness_gain_by_strategy.png", show=FAITH_SHOW_AGG_PLOTS)

# ---- deletion-curve overlay per strategy (mean across runs) ----
def _overlay_deletion(method):
    plt.figure(figsize=(9, 5))
    plotted = False
    for impl, sub in fdf.groupby("impl_name"):
        cs, qg = [], None
        for _, r in sub.iterrows():
            npz = os.path.join(str(r["run_dir"]), "xai", "faithfulness", "faithfulness_curves.npz")
            if not os.path.exists(npz):
                continue
            d = np.load(npz)
            key = f"del_exp_{method}"
            if key not in d.files:
                continue
            cs.append(d[key].mean(axis=0)); qg = d["q_grid"]
        if cs:
            plt.plot(qg, np.mean(cs, axis=0), marker="o", label=impl); plotted = True
    if plotted:
        plt.xlabel("budget q removed"); plt.ylabel("mean logit drop C_exp(q)")
        plt.title(f"Explanation-guided deletion curves ({method}) per strategy")
        plt.legend(); plt.grid(alpha=0.3)
        _finalize_plot(FAITH_OUT_DIR, f"deletion_curves_{method}_by_strategy.png", show=FAITH_SHOW_AGG_PLOTS)
    else:
        plt.close()

for _m in ("joint", "tabular", "image"):
    _overlay_deletion(_m)

print("[FAITH] aggregation done ->", FAITH_OUT_DIR)


In [ ]:
# ---------------------------------------------------------------------
# MASTER RESULTS TABLE -> artifacts/enhanced/summaries/
# One row per run + a (family,setting) mean/std/count summary. Paper Table 1
# source + AUC-vs-faithfulness trade-off scatter source.
# Requires: canonical loader cell + regenerated combined_faithfulness.csv.
# ---------------------------------------------------------------------
os.makedirs(SUMMARIES_PATH, exist_ok=True)
_frames = []
for _impl, _d in CANONICAL.items():
    _p = os.path.join(_d, "combined_summary.csv")
    if os.path.exists(_p):
        _df = pd.read_csv(_p); _df["impl_name"] = _impl; _frames.append(_df)
master = pd.concat(_frames, ignore_index=True)
_faith_csv = os.path.join(RUNS_PATH, "_faithfulness", "combined_faithfulness.csv")
if os.path.exists(_faith_csv):
    _fdf = pd.read_csv(_faith_csv)
    _keep = [c for c in ["run_dir","faithfulness_gain","faithfulness_tabular_del_gain",
             "faithfulness_image_del_gain"] if c in _fdf.columns]
    _fdf = _fdf[_keep].copy()
    # run_dir absolute prefixes differ across machines (June: /home/claudio/...,
    # now: /mnt/disk1/home/cgiovannoni/...). combined_summary.csv keeps the OLD
    # prefix, combined_faithfulness.csv the NEW one, so a raw run_dir join is
    # all-NaN. Join on the machine-independent suffix starting at "/runs/".
    def _relkey(p):
        p = str(p).replace("\\", "/")
        i = p.find("/runs/")
        return p[i:] if i >= 0 else p
    master["_key"] = master["run_dir"].map(_relkey)
    _fdf["_key"] = _fdf["run_dir"].map(_relkey)
    _fdf = _fdf.drop(columns=["run_dir"]).drop_duplicates("_key")
    _dupe = [c for c in _fdf.columns if c != "_key" and c in master.columns]
    master = master.drop(columns=_dupe).merge(_fdf, on="_key", how="left").drop(columns=["_key"])

# LORE fidelity: combined_summary.csv only carries it for families whose runner
# logged it inline. Fill vanilla + controlled_fusion from the post-hoc CSV and
# loss_regularization from each run's per-epoch CSV (last epoch, context mode --
# same semantics as the inline lore_fidelity).
import glob as _g
def _lkey(p):
    p = str(p).replace("\\", "/")
    i = p.find("/runs/")
    return p[i:] if i >= 0 else p
_lore_map = {}
_lp = os.path.join(RUNS_PATH, "_faithfulness", "lore_fidelity_posthoc.csv")
if os.path.exists(_lp):
    for _, _r in pd.read_csv(_lp).iterrows():
        _lore_map[_lkey(_r["run_dir"])] = float(_r["lore_fidelity"])
for _pe in _g.glob(os.path.join(CANONICAL["loss_regularization"], "**", "lore_fidelity_per_epoch.csv"), recursive=True):
    _e = pd.read_csv(_pe)
    if len(_e):
        _lore_map[_lkey(os.path.dirname(os.path.dirname(os.path.dirname(_pe))))] = float(_e.iloc[-1]["lore_fidelity_context"])
if _lore_map and "lore_fidelity" in master.columns:
    master["lore_fidelity"] = master["lore_fidelity"].fillna(master["run_dir"].map(_lkey).map(_lore_map))
_metrics = [c for c in ["best_val_auc","brier","ece","modality_balance","lore_fidelity",
            "faithfulness_gain","faithfulness_tabular_del_gain","faithfulness_image_del_gain"]
            if c in master.columns]
_group = [c for c in ["impl_name","setting_tag"] if c in master.columns]
summary = master.groupby(_group, dropna=False)[_metrics].agg(["mean","std","count"])
master.to_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"), index=False)
summary.to_csv(os.path.join(SUMMARIES_PATH, "master_results_summary.csv"))
print("[SUMMARY] wrote master_results_rows.csv + master_results_summary.csv ->", SUMMARIES_PATH)
display(summary)


## Low-budget faithfulness diagnosis (deletion ↔ insertion agreement)

Re-integrates the **saved** per-q curves at each `Q_MAX_REPORT` cutoff — no model calls. At the clean cutoff (`q≤0.20`) explanation-guided deletion and insertion **agree**; the divergence appears only at high budget (`q≤0.60`) — the OOD artifact (`FINDINGS_insertion_deletion_2026-07-21.md`).

> On a custom `EVAL_SET` this cell still reads the canonical `faithfulness_curves.npz` (the canonical-narrative diagnostic); it will not reflect the custom-eval numbers computed above.

In [ ]:
# Low-budget vs full-budget faithfulness from the SAVED curves (free; no GPU).
_TRAPZ = getattr(np, "trapezoid", np.trapz)

def _auc_upto(curve_mean, q_grid, q_max):
    mask = q_grid <= q_max + 1e-12
    return float(_TRAPZ(curve_mean[mask], q_grid[mask]))

def low_budget_gains(run_dir, method="joint"):
    npz = os.path.join(relmap(run_dir), "xai", "faithfulness", "faithfulness_curves.npz")
    if not os.path.exists(npz):
        return None
    d = np.load(npz); q = d["q_grid"]; res = {}
    for qmax in Q_MAX_REPORT:
        de = _auc_upto(d[f"del_exp_{method}"].mean(0), q, qmax)
        dr = _auc_upto(d[f"del_rand_{method}"].mean(0), q, qmax)
        ie = _auc_upto(d[f"ins_exp_{method}"].mean(0), q, qmax)
        ir = _auc_upto(d[f"ins_rand_{method}"].mean(0), q, qmax)
        res[qmax] = {"deletion_gain": de - dr, "insertion_gain": ie - ir}
    return res

_lb_csv = os.path.join(RUNS_PATH, "_faithfulness", "combined_faithfulness.csv")
_lb_rows = []
if os.path.exists(_lb_csv):
    for _, r in pd.read_csv(_lb_csv).iterrows():
        g = low_budget_gains(r["run_dir"], method="joint")
        if g is None:
            continue
        for qmax, gg in g.items():
            _lb_rows.append({"impl_name": r["impl_name"], "q_max": qmax, **gg})
lb = pd.DataFrame(_lb_rows)
if len(lb):
    tab = lb.groupby(["q_max", "impl_name"])[["deletion_gain", "insertion_gain"]].mean()
    tab["sign_agree"] = np.sign(tab["deletion_gain"]) == np.sign(tab["insertion_gain"])
    corr = (lb.groupby("q_max")[["deletion_gain", "insertion_gain"]]
              .apply(lambda s: s["deletion_gain"].corr(s["insertion_gain"])))
    print("deletion<->insertion gain correlation across strategies, per cutoff:")
    print(corr.to_string())
    out_csv = os.path.join(RUNS_PATH, "_faithfulness", "low_budget_faithfulness.csv")
    tab.reset_index().to_csv(out_csv, index=False)
    print("wrote", out_csv)
    display(tab)
else:
    print("[low-budget] no faithfulness_curves.npz found — run the faithfulness loop above first.")


## Appendice archiviata — verifiche Stage 0–3 (CONCLUSIONI RITIRATE)

Le verifiche Stage 0/1/2/2b/3 e i loop LORE post-hoc **non vivono più in questo notebook**:
erano 1599 righe (il 46% del file) di codice le cui conclusioni sono state ritirate.

Sono state spostate **verbatim** in **`scripts/archive_stage0_3.py`**, con l'indicazione
della cella di origine. Nulla è andato perso — il codice resta eseguibile (richiede la
sezione di setup di questo notebook già eseguita) e versionato.

**Due di quelle verifiche restano citate come fonti primarie nei report** e i loro
artefatti sono intatti:

| verifica | artefatto | citata in |
| --- | --- | --- |
| Stage 2 — attributore Permutation allineato | `runs/_robustness/stage2_aligned_attr/faithfulness_aligned_attr.csv` | REPORT_S §3.3, REPORT §7.5 |
| Stage 3 — robustezza immagine (subset + sweep σ) | `runs/_robustness/stage3_image/` | REPORT §7.4, §9 |

Perché sono state ritirate: `artifacts/enhanced/reports/FINDINGS_insertion_deletion_2026-07-21.md`.
